## **Rules from Decision Trees ( Rule Extraction and Simplification )**

A decision tree **is** a rule set: every path from the root to a leaf is a rule `IF condition AND condition … THEN class`. This notebook extracts the rules of a tree, **checks that they reproduce the tree exactly**, and then **simplifies** them as in **C4.5rules** ( dropping conditions while the *pessimistic error* does not grow, ordering the rules and adding a default class ), measuring what is gained and lost.

**Notebooks of this series:** 1. CN2 Rule Induction · 2. Expert Systems · 3. Case-Based Reasoning · 4. Apriori for Classification ( CBA ) · 5. Simple Rule Learners ( ZeroR, OneR, PRISM ) · 6. RIPPER · 7. Rules from Decision Trees · 8. RuleFit · 9. Fuzzy Rule-Based Classifier

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset and Preparation**

The same problem and the same **70/30 stratified split** of the other notebooks of the series: **malignant ( 1 ) × benign ( 0 )** tumours, the 10 *mean* attributes of the Breast Cancer dataset.

In [4]:
#Load the Breast Cancer Dataset ( Positive Class = Malignant )
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
cols = [c for c in data.data.columns if c.startswith("mean ")]            # 10 "mean" attributes
X = data.data[cols]
y = (data.target == 0).astype(int)                                        # 1 = malignant, 0 = benign
CLASS_NAMES = ["benign", "malignant"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| malignant in train:", f"{y_train.mean():.1%}")
X_train.head(3)

train: (398, 10) | test: (171, 10) | malignant in train: 37.2%


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
78,20.18,23.97,143.70,1245.0,0.129,0.345,0.375,0.160,0.291,0.081
330,16.03,15.51,105.80,793.2,0.095,0.137,0.120,0.070,0.178,0.060
378,13.66,15.15,88.27,580.6,0.083,0.075,0.042,0.025,0.179,0.059


In [ ]:
#Dictionary Used to Collect the Test Metrics of This Notebook ( the Cells Below Add Their Results to It )
results = {}

### **2. A Decision Tree and Its Rules**

The tree is limited to depth 4 and at least 5 cases per leaf so that its rules stay readable. The rules are extracted from the **fitted tree structure** and then checked against `tree.predict`.

In [ ]:
#Extraction of the Root-to-Leaf Rules, Simplification ( C4.5rules Style ) and the Ordered Rule List
from sklearn.tree import DecisionTreeClassifier, export_text

class TreeRules:
    def __init__(self, max_depth=4, min_samples_leaf=5, z=0.6745, simplify=True, seed=RANDOM_STATE):
        self.max_depth, self.min_samples_leaf, self.z, self.simplify, self.seed = max_depth, min_samples_leaf, z, simplify, seed     # z = 0.6745: 25% one-sided confidence level of C4.5

    def _mask(self, X, conds):
        m = np.ones(len(X), bool)
        for j, op, t in conds:
            m &= (X[:, j] <= t) if op == "<=" else (X[:, j] > t)
        return m

    def _pessimistic(self, errors, n):                                                  # upper confidence limit of the error rate ( Wilson )
        if n == 0:
            return 1.0
        f, z = errors / n, self.z
        return (f + z * z / (2 * n) + z * np.sqrt(f * (1 - f) / n + z * z / (4 * n * n))) / (1 + z * z / n)

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.tree_ = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf, random_state=self.seed).fit(X, y)
        t, raw = self.tree_.tree_, []
        def walk(node, conds):
            if t.children_left[node] == -1:
                raw.append(conds)
                return
            f, thr = t.feature[node], t.threshold[node]
            walk(t.children_left[node], conds + [(f, "<=", thr)]); walk(t.children_right[node], conds + [(f, ">", thr)])
        walk(0, [])
        self.raw_rules_ = [{"conds": c, "cls": int(np.bincount(y[self._mask(X, c)], minlength=2).argmax())} for c in raw]
        simplified = []
        for r in self.raw_rules_:
            conds, cls = list(r["conds"]), r["cls"]
            def err_rate(cs):
                m = self._mask(X, cs)
                return self._pessimistic(int((y[m] != cls).sum()), int(m.sum()))
            current = err_rate(conds)
            while self.simplify and len(conds) > 1:
                options = [(err_rate(conds[:i] + conds[i + 1:]), i) for i in range(len(conds))]
                best_rate, best_i = min(options)
                if best_rate <= current:
                    conds, current = conds[:best_i] + conds[best_i + 1:], best_rate
                else:
                    break
            simplified.append({"conds": conds, "cls": cls, "pess": current})
        unique = {}
        for r in simplified:
            unique.setdefault((tuple(sorted(r["conds"])), r["cls"]), r)
        ordered = sorted(unique.values(), key=lambda r: (r["pess"], -int(self._mask(X, r["conds"]).sum())))
        covered, kept = np.zeros(len(y), bool), []
        for r in ordered:                                                              # keep a rule only if it correctly classifies an uncovered case
            m = self._mask(X, r["conds"]) & ~covered
            if (y[m] == r["cls"]).any():
                r["stats"] = np.bincount(y[m], minlength=2)
                kept.append(r); covered |= m
        self.rules_ = kept
        rest = np.bincount(y[~covered], minlength=2) if (~covered).any() else np.bincount(y, minlength=2)
        self.default_ = int(rest.argmax()); self.default_p_ = (rest[1] + 1) / (rest.sum() + 2)
        return self

    def predict_raw(self, X):                                                          # the unmodified leaf rules: exactly one matches each case
        X = np.asarray(X, dtype=float)
        pred = np.full(len(X), -1)
        for r in self.raw_rules_:
            pred[self._mask(X, r["conds"])] = r["cls"]
        return pred

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        p1, free = np.full(len(X), self.default_p_), np.ones(len(X), bool)
        for r in self.rules_:
            m = self._mask(X, r["conds"]) & free
            p1[m] = (r["stats"][1] + 1) / (r["stats"].sum() + 2)
            free &= ~m
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def n_rules(self):
        return len(self.rules_)

    def rules_table(self, names):
        return pd.DataFrame([{"IF": " AND ".join(f"{names[j]} {op} {t:.4g}" for j, op, t in r["conds"]), "THEN": CLASS_NAMES[r["cls"]], "covers ( new )": int(r["stats"].sum()),
                              "precision": r["stats"][r["cls"]] / r["stats"].sum(), "length": len(r["conds"])} for r in self.rules_])

tr = TreeRules().fit(X_train, y_train)
print(export_text(tr.tree_, feature_names=[c.replace("mean ", "") for c in X_train.columns], max_depth=4)[:1800])

In [ ]:
#Quality Check 1 ( Fidelity ): Do the Extracted Leaf Rules Reproduce the Tree Exactly?
pred_tree = tr.tree_.predict(X_test.to_numpy())
pred_leaf_rules = tr.predict_raw(X_test)
print(f"leaf rules: {len(tr.raw_rules_)} | every test case matched by exactly one rule: {(pred_leaf_rules >= 0).all()} | agreement with tree.predict on the test set: {(pred_leaf_rules == pred_tree).mean():.1%} | on the training set: {(tr.predict_raw(X_train) == tr.tree_.predict(X_train.to_numpy())).mean():.1%}")

### **3. Simplified Rule List**

Each rule can lose conditions as long as its **pessimistic error** ( the upper confidence limit of the training error, as in C4.5 ) does not increase; duplicates are removed; the rules are **ordered** by pessimistic error and a rule is kept only if it correctly classifies at least one still-uncovered case. The default class is the majority of the cases left.

In [ ]:
#Evaluate the Tree and the Simplified Rule List on the Test Set
proba_tree = tr.tree_.predict_proba(X_test.to_numpy())[:, 1]
results["Decision tree ( depth 4 )"] = report_binary(y_test, pred_tree, proba_tree, "Decision tree", CLASS_NAMES, plot=False)
proba_rules = tr.predict_proba(X_test)[:, 1]
pred_rules = tr.predict(X_test)
results["Simplified rules"] = report_binary(y_test, pred_rules, proba_rules, "Simplified rules from the tree", CLASS_NAMES)
n_cond_raw = sum(len(r["conds"]) for r in tr.raw_rules_); n_cond_new = sum(len(r["conds"]) for r in tr.rules_)
print(f"leaf rules: {len(tr.raw_rules_)} with {n_cond_raw} conditions -> simplified list: {tr.n_rules()} rules with {n_cond_new} conditions + default class '{CLASS_NAMES[tr.default_]}' | agreement of the list with the tree on the test set: {(pred_rules == pred_tree).mean():.1%}")
display(pd.DataFrame(results).T[["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]].round(3))
pd.set_option("display.max_colwidth", 110)
tr.rules_table(list(X_train.columns)).round(3)

In [ ]:
#Rule Quality: Coverage and Precision of Each Rule on the Training Set and on the Test Set ( Ordered, First-Match )
rows, free_tr, free_te = [], np.ones(len(X_train), bool), np.ones(len(X_test), bool)
Xtr_np, Xte_np = X_train.to_numpy(dtype=float), X_test.to_numpy(dtype=float)
for i, r in enumerate(tr.rules_, 1):
    m_tr, m_te = tr._mask(Xtr_np, r["conds"]) & free_tr, tr._mask(Xte_np, r["conds"]) & free_te
    free_tr &= ~m_tr; free_te &= ~m_te
    rows.append({"rule": i, "length": len(r["conds"]), "train cases": int(m_tr.sum()), "train precision": (y_train.to_numpy()[m_tr] == r["cls"]).mean() if m_tr.any() else np.nan,
                 "test cases": int(m_te.sum()), "test precision": (y_test.to_numpy()[m_te] == r["cls"]).mean() if m_te.any() else np.nan})
quality = pd.DataFrame(rows)
used = quality.dropna(subset=["test precision"])
print(f"rules that decide at least one test case: {len(used)} of {len(quality)} | cases decided by the default class: train {free_tr.mean():.1%}, test {free_te.mean():.1%}")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(used["rule"] - 0.2, used["train precision"], width=0.4, color=LIGHT, label="train precision"); ax.bar(used["rule"] + 0.2, used["test precision"], width=0.4, color=RED, label="test precision")
ax.set_xlabel("rule"); ax.set_ylim(0, 1.05); ax.set_xticks(used["rule"]); ax.legend(fontsize=8, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False); plt.tight_layout(); plt.show()
quality.round(3)

### **4. Quality Metrics: Depth, Simplification and Stability**

In [ ]:
#Effect of the Tree Depth ( Repeated CV on the Training Set ): Tree vs Raw Leaf Rules vs Simplified Rule List
from sklearn.model_selection import RepeatedStratifiedKFold
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
rows = []
for depth in [2, 3, 4, 5, 6, 8]:
    acc_tree, acc_rules, n_leaves, n_rules, n_c_raw, n_c_new = [], [], [], [], [], []
    for a, b_ in cv_rep.split(X_train, y_train):
        m = TreeRules(max_depth=depth).fit(X_train.iloc[a], y_train.iloc[a])
        yb = y_train.iloc[b_].to_numpy()
        acc_tree.append((m.tree_.predict(X_train.iloc[b_].to_numpy()) == yb).mean()); acc_rules.append((m.predict(X_train.iloc[b_]) == yb).mean())
        n_leaves.append(len(m.raw_rules_)); n_rules.append(m.n_rules()); n_c_raw.append(sum(len(r["conds"]) for r in m.raw_rules_)); n_c_new.append(sum(len(r["conds"]) for r in m.rules_))
    rows.append({"max_depth": depth, "tree CV accuracy": np.mean(acc_tree), "rules CV accuracy": np.mean(acc_rules), "leaf rules": np.mean(n_leaves), "simplified rules": np.mean(n_rules), "conditions ( leaf rules )": np.mean(n_c_raw), "conditions ( simplified )": np.mean(n_c_new)})
depth_table = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
axes[0].plot(depth_table["max_depth"], depth_table["tree CV accuracy"], color=DARK, marker="o", label="decision tree"); axes[0].plot(depth_table["max_depth"], depth_table["rules CV accuracy"], color=RED, marker="o", label="simplified rules")
axes[0].set_xlabel("max_depth"); axes[0].set_ylabel("CV accuracy"); axes[0].legend(); axes[0].set_title("Accuracy vs depth")
axes[1].plot(depth_table["max_depth"], depth_table["conditions ( leaf rules )"], color=DARK, marker="o", label="leaf rules"); axes[1].plot(depth_table["max_depth"], depth_table["conditions ( simplified )"], color=RED, marker="o", label="simplified rules")
axes[1].set_xlabel("max_depth"); axes[1].set_ylabel("total conditions"); axes[1].legend(); axes[1].set_title("Size of the knowledge")
plt.tight_layout(); plt.show()
depth_table.round(3)

In [ ]:
#Reference Models on the Same Split ( Is the Technique Better than a Trivial or a Standard Model? )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

references = {"Majority class": DummyClassifier(strategy="prior"),
              "Decision tree ( depth 3 )": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
              "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))}
reference_rows = {}
for ref_name, model in references.items():
    model.fit(X_train, y_train)
    reference_rows[ref_name] = report_binary(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1], ref_name, CLASS_NAMES, plot=False)
metric_cols = ["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]
pd.concat([pd.DataFrame(results).T[metric_cols], pd.DataFrame(reference_rows).T[metric_cols]]).round(3)

In [ ]:
#Uncertainty of the Test Metrics ( Bootstrap of the Test Cases, 1,000 Resamples )
def bootstrap_ci(y_true, pred, score, n_boot=1000, seed=RANDOM_STATE):
    rng_b = np.random.default_rng(seed)
    y_true, pred, score = np.asarray(y_true), np.asarray(pred), np.asarray(score)
    acc, auc = [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        acc.append((pred[i] == y_true[i]).mean())
        auc.append(roc_auc_score(y_true[i], score[i]))
    return pd.DataFrame({"accuracy": np.percentile(acc, [2.5, 50, 97.5]), "ROC AUC": np.percentile(auc, [2.5, 50, 97.5])}, index=["2.5%", "median", "97.5%"]).T

print("the test set has", len(y_test), "cases: differences smaller than these intervals are not reliable")
bootstrap_ci(y_test, pred_rules, proba_rules).round(3)

In [ ]:
#Calibration of the Predicted Probabilities ( Brier Score and Reliability Curve )
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

proba_rules_ARRAY = np.asarray(proba_rules, dtype=float)
frac_pos, mean_pred = calibration_curve(y_test, proba_rules_ARRAY, n_bins=5, strategy="quantile")                       # bins with the same number of cases
print(f"Brier score: {brier_score_loss(y_test, proba_rules_ARRAY):.4f} | Brier score of a model that always predicts the training prevalence: {brier_score_loss(y_test, np.full(len(y_test), y_train.mean())):.4f} ( lower is better )")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", label="perfect calibration"); ax.plot(mean_pred, frac_pos, color=RED, marker="o", label="Simplified rules")
ax.set_xlabel("mean predicted probability ( 5 bins of equal size )"); ax.set_ylabel("observed share of malignant"); ax.set_title("Reliability curve"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### **5. Takeaways**

* A tree and its leaf rules are **the same model** ( the fidelity check gives 100% ): extracting rules changes the **presentation**, not the predictions
* **Simplification** removes redundant conditions: here the list keeps the same rules count with fewer conditions ( 25 instead of 39 ) and agrees with the tree on every test case; in general it can change some predictions, because the generalised rules cover cases of other leaves, so always measure the agreement
* Rules of a tree are **mutually exclusive** by construction, while the simplified list is **ordered** and relies on first-match semantics
* The depth controls the size: see how many conditions the knowledge needs at each depth and whether the cross-validated accuracy keeps improving; the depth table is a sensitivity analysis, not a tuning procedure
* Rule extraction is also the way to **explain** more complex models: see *RuleFit* ( notebook 8 ), where the rules come from a tree **ensemble**